# Exploratory Data Analysis: NovaCRM SaaS Metrics

**Project 04 -- Executive KPI Report** | Notebook 2 of 5

## Purpose

This notebook performs a thorough exploratory data analysis on NovaCRM's 24 months of SaaS metrics. Before building anomaly detection models or forecasting pipelines, we need to understand the data's shape, distributions, correlations, and temporal patterns.

### Key Questions

- What do the distributions of each KPI look like? Are there outliers or skew?
- Which metrics are most correlated with each other?
- Can we decompose MRR into trend, seasonal, and residual components?
- How do segments differ in their KPI trajectories?
- What monthly and quarterly patterns emerge?

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Color palette
COLORS = {
    "cyan": "#06B6D4", "violet": "#8B5CF6", "emerald": "#10B981",
    "amber": "#F59E0B", "rose": "#F43F5E",
}
SEGMENT_COLORS = {
    "Starter": COLORS["cyan"],
    "Professional": COLORS["violet"],
    "Enterprise": COLORS["emerald"],
}

# Load all datasets
monthly = pd.read_parquet("../data/processed/monthly_metrics.parquet")
segments = pd.read_parquet("../data/processed/segment_metrics.parquet")
monthly_kpis = pd.read_parquet("../data/processed/monthly_kpis.parquet")

# Add computed columns
monthly["month_str"] = monthly["month"].dt.strftime("%Y-%m")
monthly["quarter"] = monthly["month"].dt.quarter
monthly["year"] = monthly["month"].dt.year

print(f"Monthly metrics: {monthly.shape}")
print(f"Segment metrics: {segments.shape}")
print(f"Monthly KPIs (with derived): {monthly_kpis.shape}")

## 1. Dataset Overview

Let's start with a statistical summary of all core metrics to understand central tendency, spread, and potential issues (missing values, extreme outliers).

In [ ]:
# Summary statistics for core metrics
core_metrics = [
    "mrr", "arr", "new_mrr", "expansion_mrr", "churned_mrr", "contraction_mrr",
    "total_customers", "new_customers", "churned_customers",
    "logo_churn_rate", "revenue_churn_rate", "nrr",
    "cac", "ltv", "ltv_cac_ratio", "payback_months",
    "dau", "mau", "dau_mau_ratio", "feature_adoption_rate",
    "support_tickets", "avg_resolution_hours", "nps",
    "gross_margin", "burn_rate", "runway_months", "rule_of_40",
]

summary = monthly[core_metrics].describe().T
summary["cv"] = summary["std"] / summary["mean"]  # coefficient of variation
summary = summary[["count", "mean", "std", "cv", "min", "25%", "50%", "75%", "max"]]
print("Core Metrics Summary (24 months):")
print("=" * 100)
print(summary.to_string())
print(f"\nMissing values: {monthly[core_metrics].isnull().sum().sum()}")

## 2. Distribution Analysis

Understanding the shape of each KPI distribution helps us choose appropriate statistical methods later (e.g., z-score anomaly detection assumes approximate normality).

### Revenue Metrics

In [ ]:
# Revenue metric distributions -- histograms + box plots
rev_metrics = ["mrr", "new_mrr", "expansion_mrr", "churned_mrr", "contraction_mrr"]
rev_labels = ["MRR", "New MRR", "Expansion MRR", "Churned MRR", "Contraction MRR"]

fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=[f"{l} Distribution" for l in rev_labels] + [f"{l} Box" for l in rev_labels],
    row_heights=[0.6, 0.4],
    vertical_spacing=0.15,
)

palette = [COLORS["cyan"], COLORS["emerald"], COLORS["violet"], COLORS["rose"], COLORS["amber"]]

for i, (metric, label) in enumerate(zip(rev_metrics, rev_labels)):
    fig.add_trace(go.Histogram(
        x=monthly[metric], nbinsx=8,
        marker_color=palette[i], opacity=0.8, name=label, showlegend=False,
    ), row=1, col=i+1)
    fig.add_trace(go.Box(
        y=monthly[metric], marker_color=palette[i],
        name=label, showlegend=False,
    ), row=2, col=i+1)

fig.update_layout(
    height=500, template="plotly_white",
    title="Revenue Metrics: Distributions and Box Plots",
    font=dict(family="Inter, sans-serif"),
)
fig.show()

# Check skewness
from scipy import stats as sp_stats
print("\nSkewness of revenue metrics:")
for metric, label in zip(rev_metrics, rev_labels):
    skew = sp_stats.skew(monthly[metric].dropna())
    print(f"  {label:20s}: {skew:+.3f}  {'(left-skewed)' if skew < -0.5 else '(right-skewed)' if skew > 0.5 else '(approx. symmetric)'}")

### Unit Economics and Engagement Distributions

These metrics directly inform go-to-market efficiency and product health.

In [ ]:
# Unit economics and engagement distributions
unit_metrics = [
    ("cac", "CAC ($)"),
    ("ltv", "LTV ($)"),
    ("ltv_cac_ratio", "LTV:CAC Ratio"),
    ("payback_months", "Payback (months)"),
    ("dau_mau_ratio", "DAU/MAU Ratio"),
    ("feature_adoption_rate", "Feature Adoption"),
    ("nps", "NPS Score"),
    ("gross_margin", "Gross Margin"),
]

fig = make_subplots(rows=2, cols=4, subplot_titles=[m[1] for m in unit_metrics])

palette2 = [COLORS["cyan"], COLORS["emerald"], COLORS["violet"], COLORS["amber"],
            COLORS["rose"], COLORS["cyan"], COLORS["violet"], COLORS["emerald"]]

for idx, (col, label) in enumerate(unit_metrics):
    row = idx // 4 + 1
    col_pos = idx % 4 + 1
    fig.add_trace(go.Histogram(
        x=monthly[col], nbinsx=8,
        marker_color=palette2[idx], opacity=0.8, showlegend=False,
    ), row=row, col=col_pos)

fig.update_layout(
    height=450, template="plotly_white",
    title="Unit Economics & Engagement Distributions",
    font=dict(family="Inter, sans-serif"),
)
fig.show()

## 3. Correlation Analysis

Understanding correlations between metrics reveals:
- **Expected relationships** (e.g., MRR and total_customers should be positively correlated)
- **Leading indicators** (e.g., does NPS lead churn changes?)
- **Redundant metrics** (high correlation suggests we can simplify reporting)

In [ ]:
# Correlation matrix for key metrics
corr_metrics = [
    "mrr", "nrr", "logo_churn_rate", "revenue_churn_rate", "nps",
    "total_customers", "cac", "ltv", "ltv_cac_ratio",
    "dau_mau_ratio", "feature_adoption_rate", "support_tickets",
    "avg_resolution_hours", "gross_margin", "rule_of_40",
]

corr_matrix = monthly[corr_metrics].corr()

# Short labels for readability
short_labels = [
    "MRR", "NRR", "Logo Churn", "Rev Churn", "NPS",
    "Customers", "CAC", "LTV", "LTV:CAC",
    "DAU/MAU", "Adoption", "Tickets",
    "Resolution (h)", "Gross Margin", "Rule of 40",
]

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=short_labels,
    y=short_labels,
    colorscale=[
        [0.0, COLORS["rose"]],
        [0.5, "#FFFFFF"],
        [1.0, COLORS["cyan"]],
    ],
    zmid=0,
    text=[[f"{v:.2f}" for v in row] for row in corr_matrix.values],
    texttemplate="%{text}",
    textfont=dict(size=9),
    colorbar=dict(title="Correlation"),
))

fig.update_layout(
    title="Correlation Matrix: Key SaaS Metrics",
    height=650,
    width=750,
    template="plotly_white",
    font=dict(family="Inter, sans-serif"),
    xaxis_tickangle=45,
)
fig.show()

# Top positive and negative correlations
import itertools
pairs = []
for i, j in itertools.combinations(range(len(corr_metrics)), 2):
    pairs.append((short_labels[i], short_labels[j], corr_matrix.iloc[i, j]))
pairs.sort(key=lambda x: abs(x[2]), reverse=True)

print("\nTop 5 strongest correlations:")
for a, b, r in pairs[:5]:
    direction = "positive" if r > 0 else "negative"
    print(f"  {a} <-> {b}: r={r:+.3f} ({direction})")

print("\nTop 5 weakest correlations:")
for a, b, r in pairs[-5:]:
    print(f"  {a} <-> {b}: r={r:+.3f}")

## 4. Time-Series Decomposition of MRR

Decomposing MRR into trend, seasonal, and residual components helps us understand what drives revenue movement and how much variance comes from predictable patterns vs. noise.

> **So What?** If seasonality explains a large portion of MRR variance, the forecasting model (notebook 04) should explicitly model it. If residuals dominate, the business is driven more by event-based disruptions.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

# Set month as index for decomposition
mrr_series = monthly.set_index("month")["mrr"]

# Since we have 24 months, use period=12 for annual seasonality
decomp = seasonal_decompose(mrr_series, model="additive", period=12)

fig = make_subplots(
    rows=4, cols=1,
    subplot_titles=("Observed MRR", "Trend Component", "Seasonal Component", "Residual (Noise)"),
    shared_xaxes=True,
    vertical_spacing=0.08,
)

fig.add_trace(go.Scatter(
    x=mrr_series.index, y=mrr_series.values,
    mode="lines+markers", line=dict(color=COLORS["cyan"], width=2), name="Observed",
), row=1, col=1)

fig.add_trace(go.Scatter(
    x=decomp.trend.index, y=decomp.trend.values,
    mode="lines", line=dict(color=COLORS["violet"], width=2), name="Trend",
), row=2, col=1)

fig.add_trace(go.Bar(
    x=decomp.seasonal.index, y=decomp.seasonal.values,
    marker_color=COLORS["emerald"], name="Seasonal",
), row=3, col=1)

fig.add_trace(go.Scatter(
    x=decomp.resid.index, y=decomp.resid.values,
    mode="markers+lines", line=dict(color=COLORS["rose"], width=1),
    marker=dict(size=5), name="Residual",
), row=4, col=1)

fig.update_layout(
    height=700, template="plotly_white", showlegend=False,
    title="MRR Additive Decomposition (period=12)",
    font=dict(family="Inter, sans-serif"),
)
for i in range(1, 5):
    fig.update_yaxes(tickprefix="$", tickformat=",", row=i, col=1)
fig.show()

# Variance decomposition
total_var = mrr_series.var()
trend_var = decomp.trend.dropna().var()
seasonal_var = decomp.seasonal.var()
resid_var = decomp.resid.dropna().var()
print(f"\nVariance decomposition:")
print(f"  Trend explains:    {trend_var / total_var:.1%}")
print(f"  Seasonal explains: {seasonal_var / total_var:.1%}")
print(f"  Residual:          {resid_var / total_var:.1%}")

## 5. Segment Deep-Dive

Comparing how Starter, Professional, and Enterprise segments evolve over time reveals whether growth is balanced or concentrated.

In [ ]:
# Segment comparison: key metrics over time
seg_metrics = [
    ("mrr", "MRR ($)", True),
    ("logo_churn_rate", "Logo Churn Rate", False),
    ("nps", "NPS Score", False),
    ("ltv_cac_ratio", "LTV:CAC Ratio", False),
]

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[m[1] for m in seg_metrics],
    shared_xaxes=True,
)

for idx, (metric, label, is_currency) in enumerate(seg_metrics):
    row = idx // 2 + 1
    col = idx % 2 + 1
    for seg in ["Starter", "Professional", "Enterprise"]:
        seg_data = segments[segments["segment"] == seg]
        fig.add_trace(go.Scatter(
            x=seg_data["month"], y=seg_data[metric],
            mode="lines", name=seg if idx == 0 else None,
            line=dict(color=SEGMENT_COLORS[seg], width=2),
            legendgroup=seg, showlegend=(idx == 0),
        ), row=row, col=col)

    if is_currency:
        fig.update_yaxes(tickprefix="$", tickformat=",", row=row, col=col)
    elif "rate" in metric:
        fig.update_yaxes(tickformat=".1%", row=row, col=col)

fig.update_layout(
    height=550, template="plotly_white",
    title="Segment Trajectories: Key Metrics Over Time",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0.5, xanchor="center"),
    font=dict(family="Inter, sans-serif"),
)
fig.show()

### Segment Insights

> **So What?**
> - Enterprise MRR grows steadily and shows the least volatility -- this is the anchor segment
> - Starter churn is consistently 2--3x higher than Enterprise, confirming the need for segment-specific retention strategies
> - NPS divergence between segments widened after the pricing event, suggesting Starter customers were disproportionately impacted
> - Enterprise LTV:CAC consistently exceeds 5x, making it the most capital-efficient segment

In [ ]:
# MRR composition: stacked area by segment
fig = go.Figure()

for seg in ["Starter", "Professional", "Enterprise"]:
    seg_data = segments[segments["segment"] == seg].sort_values("month")
    fig.add_trace(go.Scatter(
        x=seg_data["month"], y=seg_data["mrr"],
        mode="lines", name=seg,
        line=dict(width=0.5, color=SEGMENT_COLORS[seg]),
        stackgroup="one",
        fillcolor=SEGMENT_COLORS[seg],
    ))

fig.update_layout(
    title="MRR Composition by Segment (Stacked Area)",
    xaxis_title="Month",
    yaxis_title="MRR ($)",
    yaxis_tickprefix="$",
    yaxis_tickformat=",",
    template="plotly_white",
    height=450,
    font=dict(family="Inter, sans-serif"),
)
fig.show()

## 6. Monthly and Quarterly Patterns

Analyzing patterns by calendar month and quarter helps identify systematic effects useful for budgeting and resource planning.

In [ ]:
# Quarterly patterns
quarterly = monthly.copy()
quarterly["quarter_label"] = quarterly["month"].dt.to_period("Q").astype(str)

fig = make_subplots(rows=1, cols=3, subplot_titles=(
    "New MRR by Quarter", "Churned Customers by Quarter", "NPS by Quarter",
))

q_colors = [COLORS["emerald"], COLORS["cyan"], COLORS["amber"], COLORS["rose"]]

for col_idx, (metric, label) in enumerate([
    ("new_mrr", "New MRR"), ("churned_customers", "Churned"), ("nps", "NPS")
]):
    for qi, q in enumerate(sorted(quarterly["quarter"].unique())):
        q_data = quarterly[quarterly["quarter"] == q]
        fig.add_trace(go.Box(
            y=q_data[metric],
            name=f"Q{q}",
            marker_color=q_colors[qi],
            legendgroup=f"Q{q}",
            showlegend=(col_idx == 0),
        ), row=1, col=col_idx+1)

fig.update_layout(
    height=400, template="plotly_white",
    title="Quarterly Patterns in Key Metrics",
    font=dict(family="Inter, sans-serif"),
)
fig.update_yaxes(tickprefix="$", tickformat=",", row=1, col=1)
fig.show()

## 7. MRR Waterfall Analysis

The MRR waterfall decomposes monthly revenue changes into their components: new business, expansion, contraction, and churn. This is the single most important chart for SaaS revenue understanding.

In [ ]:
# MRR waterfall for each month
latest_idx = monthly.index[-1]
latest = monthly.iloc[latest_idx]
prev = monthly.iloc[latest_idx - 1]

starting_mrr = prev["mrr"]
components = ["New MRR", "Expansion", "Contraction", "Churned", "Ending MRR"]
values = [
    latest["new_mrr"],
    latest["expansion_mrr"],
    -latest["contraction_mrr"],
    -latest["churned_mrr"],
    latest["mrr"],
]
measures = ["relative", "relative", "relative", "relative", "total"]

fig = go.Figure(go.Waterfall(
    x=["Starting MRR"] + components,
    y=[starting_mrr] + values,
    measure=["absolute"] + measures,
    connector=dict(line=dict(color="#d1d5db")),
    increasing=dict(marker=dict(color=COLORS["emerald"])),
    decreasing=dict(marker=dict(color=COLORS["rose"])),
    totals=dict(marker=dict(color=COLORS["cyan"])),
    text=[f"${v:,.0f}" for v in [starting_mrr] + values],
    textposition="outside",
))

fig.update_layout(
    title=f"MRR Waterfall -- {latest['month'].strftime('%B %Y')}",
    yaxis_title="MRR ($)",
    yaxis_tickprefix="$",
    yaxis_tickformat=",",
    template="plotly_white",
    height=450,
    font=dict(family="Inter, sans-serif"),
)
fig.show()

net_new = latest["new_mrr"] + latest["expansion_mrr"] - latest["contraction_mrr"] - latest["churned_mrr"]
print(f"\nNet New MRR: ${net_new:,.0f}")
print(f"  New business:  ${latest['new_mrr']:,.0f} ({latest['new_mrr']/starting_mrr:.1%} of starting)")
print(f"  Expansion:     ${latest['expansion_mrr']:,.0f} ({latest['expansion_mrr']/starting_mrr:.1%})")
print(f"  Contraction:  -${latest['contraction_mrr']:,.0f} ({latest['contraction_mrr']/starting_mrr:.1%})")
print(f"  Churned:      -${latest['churned_mrr']:,.0f} ({latest['churned_mrr']/starting_mrr:.1%})")

In [ ]:
# MRR components over time (stacked bar)
fig = go.Figure()

fig.add_trace(go.Bar(
    x=monthly["month"], y=monthly["new_mrr"],
    name="New MRR", marker_color=COLORS["emerald"],
))
fig.add_trace(go.Bar(
    x=monthly["month"], y=monthly["expansion_mrr"],
    name="Expansion", marker_color=COLORS["cyan"],
))
fig.add_trace(go.Bar(
    x=monthly["month"], y=-monthly["contraction_mrr"],
    name="Contraction", marker_color=COLORS["amber"],
))
fig.add_trace(go.Bar(
    x=monthly["month"], y=-monthly["churned_mrr"],
    name="Churned", marker_color=COLORS["rose"],
))

fig.update_layout(
    barmode="relative",
    title="MRR Movement Components Over Time",
    xaxis_title="Month",
    yaxis_title="MRR Change ($)",
    yaxis_tickprefix="$",
    yaxis_tickformat=",",
    template="plotly_white",
    height=450,
    font=dict(family="Inter, sans-serif"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0.5, xanchor="center"),
)
fig.show()

## 8. Composite Health Score

The KPI pipeline computes a weighted health score (0--100) combining six dimensions:

| Component | Weight | Good Threshold | Bad Threshold |
|-----------|--------|---------------|--------------|
| MRR Growth | 20% | >= 5% MoM | <= 0% |
| NRR | 20% | >= 115% | <= 95% |
| Churn | 15% | <= 2% | >= 5% |
| NPS | 15% | >= 55 | <= 25 |
| Rule of 40 | 15% | >= 40% | <= 15% |
| LTV:CAC | 15% | >= 4.0x | <= 1.5x |

In [ ]:
# Health score over time
fig = go.Figure()

# Color code by status
status_colors = {"green": COLORS["emerald"], "yellow": COLORS["amber"], "red": COLORS["rose"]}
bar_colors = [status_colors.get(s, "gray") for s in monthly_kpis["health_status"]]

fig.add_trace(go.Bar(
    x=monthly_kpis["month"],
    y=monthly_kpis["health_score"],
    marker_color=bar_colors,
    text=[f"{v:.0f}" for v in monthly_kpis["health_score"]],
    textposition="outside",
))

fig.add_hline(y=75, line_dash="dash", line_color=COLORS["emerald"],
              annotation_text="Green threshold (75)")
fig.add_hline(y=50, line_dash="dash", line_color=COLORS["amber"],
              annotation_text="Yellow threshold (50)")

fig.update_layout(
    title="NovaCRM Composite Health Score Over Time",
    xaxis_title="Month",
    yaxis_title="Health Score (0-100)",
    yaxis_range=[0, 105],
    template="plotly_white",
    height=400,
    font=dict(family="Inter, sans-serif"),
)
fig.show()

# Health status distribution
status_counts = monthly_kpis["health_status"].value_counts()
print("\nHealth Status Distribution:")
for status, count in status_counts.items():
    print(f"  {status.upper():8s}: {count} months ({count/len(monthly_kpis):.0%})")

## EDA Summary

### Key Findings

1. **MRR shows strong upward trend** with clear seasonality -- Q1 strongest, December weakest
2. **Churn and NPS are moderately anti-correlated** (r ~ -0.4), confirming NPS as a leading indicator of retention
3. **Trend dominates MRR variance** over seasonal and residual components, suggesting the growth engine is working
4. **Enterprise segment is the stability anchor** -- lowest churn, highest NPS, best unit economics
5. **The pricing event (Q3 2024) is clearly visible** across churn, NPS, and support metrics
6. **Health score drops to yellow/red during the pricing event** then recovers -- validating the composite scoring model

### Next Steps

- **Notebook 03**: Anomaly detection will formally flag the pricing event and product launch
- **Notebook 04**: Forecasting will project MRR and churn trajectories with confidence intervals
- **Notebook 05**: Report automation will assemble these insights into an executive PDF